# Anomaly detection: training launcher


## 1. Preparation

In [ ]:
# Mount the drive
from google.colab import drive
drive.mount('/content/drive')

# Clone the repository code
!git clone https://github.com/Fabio-Feruglio/Jet-Image-Tagging.git

# Move to the correct folder
%cd Jet-Image-Tagging/anomaly_detection

In [ ]:
# Install dependencies
!pip install optuna wandb

# Authenticate on WandB
from google.colab import userdata
import wandb
wandb.login(key=userdata.get('WANDB_API_KEY'))

In [ ]:
# TensorBoard loader
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/JetTagging/checkpoints/tensorboard_logs_anomaly_detection

In [ ]:
# Copy the dataset from Drive to local storage
!cp /content/drive/MyDrive/JetTagging/data/jet_images_128.h5 /content/jet_images_128.h5
print("Dataset copied to local storage.")

## 2. Tag and Train, or double AE system

In [ ]:
# Training
!python /content/Jet-Image-Tagging/anomaly_detection/src/train_tnt.py \
    --data_path "/content/jet_images_128.h5" \
    --save_dir_ae1 "/content/drive/MyDrive/JetTagging/anomaly_detection/checkpoints_ae1" \
    --save_dir_ae2 "/content/drive/MyDrive/JetTagging/anomaly_detection/checkpoints_ae2" \
    --epochs_ae1 40 \
    --epochs_ae2 50 \
    --latent_dim_ae1 16 \
    --latent_dim_ae2 32 \
    --batch_size 64 \
    --lr 0.0001 \
    --bg_classes 0 1 \
    --num_train_samples 30000 \
    --noise_prob 0.1 \
    --threshold_percentile 90.0 \
    --skip_ae1

In [ ]:
#Evaluation
!python /content/Jet-Image-Tagging/anomaly_detection/src/evaluation_tnt.py \
    --model_dir_ae1 "/content/drive/MyDrive/JetTagging/anomaly_detection/checkpoints_ae1" \
    --model_dir_ae2 "/content/drive/MyDrive/JetTagging/anomaly_detection/checkpoints_ae2" \
    --num_train_samples 30000 \
    --latent_dim_ae1 16 \
    --latent_dim_ae2 32 \
    --batch_size 64 \
    --img_size 128 \
    --bg_classes 0 1 \
    --data_path "/content/jet_images_128.h5" \
    --save_dir "/content/drive/MyDrive/JetTagging/anomaly_detection/results_tnt"

## 3 Symmetric autoencoder with pepper noise training

In [ ]:
# Train the Denoising Symmetric Autoencoder (10% Pepper Noise)
!python /content/Jet-Image-Tagging/anomaly_detection/src/train_symmetricAdaGrad.py \
    --data_path "/content/jet_images_128.h5" \
    --save_dir "/content/drive/MyDrive/JetTagging/anomaly_detection/checkpoints/symmetric_AdaGrad" \
    --epochs 20 \
    --latent_space_dim 16 \
    --batch_size 64 \
    --lr 0.0001 \
    --bg_classes 0 1 \
    --img_size 128 \
    --weight_decay 0.0001 \
    --max_samples 30000 \
    --patience 10 \
    --noise_prob 0.1

In [ ]:
# Evaluation - Denoising Autoencoder
!python /content/Jet-Image-Tagging/anomaly_detection/src/evaluation_symmetricAE.py \
    --model_path "/content/drive/MyDrive/JetTagging/anomaly_detection/checkpoints/symmetric_AdaGrad/autoencoder_best.pth" \
    --max_samples 30000 \
    --latent_space_dim 16 \
    --batch_size 64 \
    --img_size 128 \
    --bg_classes 0 1 \
    --data_path "/content/jet_images_128.h5" \
    --save_dir "/content/drive/MyDrive/JetTagging/anomaly_detection/results/symmetric_AdaGrad"